In [1]:
%%capture
%pip install -U \
langchain \
langchain-core \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-chroma \
langchain-classic \
langchain-ibm \
ibm-watsonx-ai \
chromadb \
sentence-transformers \
transformers \
huggingface-hub \
wget

In [1]:
%pip install --upgrade \
numpy \
pandas \
scipy \
scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 82.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
  Attempting uninstall: scikit-learn
    Found existing installation: sciki

In [10]:
import warnings
warnings.filterwarnings("ignore")

import wget

# LangChain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

# Local Hugging Face LLM
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

print("✅ All imports successful!")

✅ All imports successful!


In [11]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)
print('file downloaded')

file downloaded


In [12]:
with open(filename, 'r') as file:
    # Read the contents of the file
    contents = file.read()
    print(contents)

1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violations of 

In [13]:
llm_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    max_new_tokens=256,
    temperature=0.3,
    clean_up_tokenization_spaces=False
)

llm = HuggingFacePipeline(pipeline=llm_pipeline)

print("✅ Local LLM loaded!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Local LLM loaded!


In [14]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model ready!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model ready!


In [15]:
loader = TextLoader(filename)

documents = loader.load()

print("Number of documents:", len(documents))
print(documents[0].page_content[:400])

Number of documents: 1
1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues


In [16]:
text_splitter = CharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print(chunks[0].page_content)

Total chunks: 18
1.	Code of Conduct


In [18]:
import shutil
import os
from langchain_chroma import Chroma

# Delete old Chroma database if it exists
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

# Create a fresh vector database
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ Fresh Chroma vector database created!")

✅ Fresh Chroma vector database created!


In [20]:
query = "What is the mobile policy?"

results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content[:250])


--- Retrieved Chunk 1 ---
4.	Mobile Phone Policy

--- Retrieved Chunk 2 ---
The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that employees utilize mobile phones in a manner consiste

--- Retrieved Chunk 3 ---
3.	Internet and Email Policy


In [22]:
from langchain_classic.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm,                      # Local Qwen model
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k":3}),
    return_source_documents=False
)

print("✅ RetrievalQA chain created!")

✅ RetrievalQA chain created!


In [23]:
query = "Can you summarize the document for me?"

response = qa.invoke({"query": query})

print(response["result"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

1.	Code of Conduct

2.	Recruitment Policy

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We 

In [24]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

# Custom RAG prompt
prompt_template = """
Use the information from the document to answer the question at the end.
If you don't know the answer, just say that you don't know.
Definitely do not try to make up an answer.

Context:
{context}

Question: {question}

Answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

# Create RetrievalQA chain using our LOCAL vector database
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    chain_type_kwargs=chain_type_kwargs,
    return_source_documents=False
)

print("✅ Custom RetrievalQA created!")

✅ Custom RetrievalQA created!


In [29]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

llm_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    max_new_tokens=120,      # was 256
    temperature=0.2,
    do_sample=False,         # deterministic, faster
    clean_up_tokenization_spaces=False
)

llm = HuggingFacePipeline(pipeline=llm_pipeline)

print("✅ Faster local LLM loaded!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Faster local LLM loaded!


In [25]:
query = "Can I eat in company vehicles?"

response = qa.invoke({"query": query})

print(response["result"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the information from the document to answer the question at the end.
If you don't know the answer, just say that you don't know.
Definitely do not try to make up an answer.

Context:
Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. This policy is in place to ensure a safe and healthy environment for all employees, visitors, and the general public.
Designated Smoking Areas: Smoking is only permitted in designated smoking areas, as marked by appropriate signage. These areas have been chosen to minimize exposure to secondhand smoke and to maintain the overall cleanliness of the premises.
Smoking Restrictions: Smoking inside company buildings, offices, meeting rooms, and other enclosed spaces is strictly prohibited. This includes electronic cigarettes and vaping devices.
Compliance with Applicable Laws: All employees and visitors must adhere to relevant federal, state, and local smoking laws and 

In [26]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain

# Conversation memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Conversational Retrieval Chain
qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    memory=memory,
    return_source_documents=False,
    get_chat_history=lambda h: h
)

print("✅ Conversational RAG chain created!")

✅ Conversational RAG chain created!


In [27]:
history = []

# Question 1
query = "What is the mobile policy?"

result = qa.invoke({"question": query})

print("Q1:", query)
print("A1:", result["answer"])

history.append((query, result["answer"]))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q1: What is the mobile policy?
A1: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

4.	Mobile Phone Policy

The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance.
Acceptable Use: Mobile devices are primarily intended for work-related tasks. Limited personal usage is allowed, provided it does not disrupt work obligations.
Security: Safeguard your mobile device and access credentials. Exercise caution when downloading apps or clicking links from unfamiliar sources. Promptly report security concerns or suspicious activities related to your mobile device.
Confidentiality: Avoid transmitting sensitive company information via unsecured messagi

In [30]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k":2}),  # retrieve only 2 chunks
    memory=memory,
    return_source_documents=False,
    get_chat_history=lambda h: h
)

In [31]:
result = qa.invoke({"question": "What is the mobile policy?"})
print(result["answer"])

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

4.	Mobile Phone Policy

The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance.
Acceptable Use: Mobile devices are primarily intended for work-related tasks. Limited personal usage is allowed, provided it does not disrupt work obligations.
Security: Safeguard your mobile device and access credentials. Exercise caution when downloading apps or clicking links from unfamiliar sources. Promptly report security concerns or suspicious activities related to your mobile device.
Confidentiality: Avoid transmitting sensitive company information via unsecured messaging apps or emails. Be discreet when

In [35]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

llm_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    max_new_tokens=120,
    temperature=0.2,
    do_sample=False,
    return_full_text=False,      # ⭐ IMPORTANT FIX
    clean_up_tokenization_spaces=False
)

llm = HuggingFacePipeline(pipeline=llm_pipeline)

print("✅ Fixed local LLM loaded!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Fixed local LLM loaded!


In [36]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain

def qa_chat():
    # Conversation memory
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    # Conversational RAG chain using local Chroma vectorstore
    qa = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
        memory=memory,
        get_chat_history=lambda h: h,
        return_source_documents=False
    )

    print("🤖 Company Policy Chatbot")
    print("Type 'bye', 'exit' or 'quit' to stop.\n")

    while True:
        query = input("Question: ")

        if query.lower() in ["quit", "exit", "bye"]:
            print("Answer: Goodbye!")
            break

        # New LangChain syntax
        result = qa.invoke({"question": query})

        print("Answer:", result["answer"])
        print("-" * 60)

In [37]:
qa_chat()

🤖 Company Policy Chatbot
Type 'bye', 'exit' or 'quit' to stop.

Question: what is the smoking policy?


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:  The smoking policy states that smoking is only allowed in designated smoking areas, where appropriate signage indicates it's forbidden. It also prohibits smoking inside company buildings, offices, meeting rooms, and enclosed spaces, including electronic cigarettes and vaping devices. Employees and visitors are required to comply with applicable laws and regulations regarding smoking. Non-compliance can result in disciplinary action, such as fines or termination of employment. Regular reviews of the policy are conducted to keep it current. The policy aims to create a smoke-free and safe work environment for all employees. (Source: [Company Website])
------------------------------------------------------------
Question: can u list some points?


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:  To summarize, the smoking policy aims to establish guidelines for smoking behavior on company premises, including designated smoking areas, restrictions on smoking indoors, compliance with applicable laws, proper disposal of smoking materials, no smoking in company vehicles, and periodic review of policies. However, specific areas or types of smoking are not mentioned in the given context. Please provide more details to get a comprehensive understanding of the smoking policy. [End]
------------------------------------------------------------
Question: exit
Answer: Goodbye!
